In [16]:
import cv2
import numpy as np
import tensorflow as tf
import os

In [17]:
from keras import models
from keras import layers
from keras import optimizers
from keras.preprocessing.image import ImageDataGenerator

In [18]:
Imagesize=64
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(Imagesize,Imagesize,3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))
model.add(layers.Conv2D(256, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D(2, 2))
model.add(layers.Flatten())
model.add(layers.Dropout(0.25))
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(25, activation='softmax'))

In [19]:
import keras

In [20]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1, min_lr=0.0001)
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=0, patience=2, verbose=0, mode='auto')

In [21]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 62, 62, 32)        896       
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 31, 31, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 29, 29, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 14, 14, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_6 (Conv2D)           (None, 12, 12, 128)       73856     
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 6, 6, 128)        

In [22]:
data_train = ImageDataGenerator(rescale=1.0/255.0,rotation_range=40,width_shift_range=0.2,height_shift_range=0.2,
                                shear_range=0.2,zoom_range=0.2,horizontal_flip=True)
data_valid = ImageDataGenerator(rescale=1.0/255.0)

In [23]:
batch = 96

In [24]:
train_gen = data_train.flow_from_directory('train_processed',target_size=(Imagesize,Imagesize),batch_size=batch, 
                                           class_mode='categorical')

test_gen = data_valid.flow_from_directory('test_processed', target_size=(Imagesize,Imagesize), batch_size=batch, 
                                           class_mode='categorical')

Found 7999 images belonging to 25 classes.
Found 2001 images belonging to 25 classes.


In [25]:
model.fit_generator(train_gen, epochs=50, steps_per_epoch=50, validation_data = test_gen, 
                    validation_steps=4, callbacks=[reduce_lr, early_stop])

Epoch 1/50


C:\Users\Asim\AppData\Local\Temp\ipykernel_37344\1847944093.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  model.fit_generator(train_gen, epochs=50, steps_per_epoch=50, validation_data = test_gen,


50/50 [==============================] - 4s 75ms/step - loss: 2.7679 - accuracy: 0.1704 - val_loss: 1.1837 - val_accuracy: 0.6849 - lr: 0.0010
Epoch 2/50
50/50 [==============================] - 4s 74ms/step - loss: 1.7341 - accuracy: 0.4479 - val_loss: 0.3462 - val_accuracy: 0.9297 - lr: 0.0010
Epoch 3/50
50/50 [==============================] - 4s 76ms/step - loss: 1.0734 - accuracy: 0.6562 - val_loss: 0.1403 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 4/50
50/50 [==============================] - 4s 74ms/step - loss: 0.6001 - accuracy: 0.8076 - val_loss: 0.0590 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 5/50
50/50 [==============================] - 4s 74ms/step - loss: 0.4144 - accuracy: 0.8669 - val_loss: 0.0351 - val_accuracy: 0.9896 - lr: 0.0010
Epoch 6/50
50/50 [==============================] - 4s 74ms/step - loss: 0.3276 - accuracy: 0.8942 - val_loss: 0.0164 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 7/50
50/50 [==============================] - 4s 73ms/step - loss: 0.2242 - accur

In [26]:
#to create a saved model, use the following command
model.save('islcnnmodel.h5')

In [27]:
imgs,labels=next(test_gen)
scores = model.evaluate(imgs,labels,verbose=0)
print("Accuracy: ",scores[1])

Accuracy:  1.0
